In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#Load Files
data_folder = Path("data")

csv_files = sorted(data_folder.glob("*.csv"))

print("Number of CSV files found:", len(csv_files))
csv_files

#Combine all files
dfs = []

for file in csv_files:
    temp_df = pd.read_csv(file, low_memory=False)
    temp_df["SourceFile"] = file.name
    dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)

print("Combined shape:", df.shape)
df.head()

Number of CSV files found: 6
Combined shape: (124404, 79)


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,SourceFile
0,ContraCosta,ContraCosta,"Carpet,Tile,Wood",NaN,NaN,NaN,False,1998000.0,1150041639,teresa@teresahooper.com,...,10080.0,NaN,False,3.0,San Ramon Valley,94596,975.0,10080.0,NaN,CRMLSSold202512.csv
1,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,NaN,1150038872,homes@roomsantacruz.com,...,5663.0,NaN,NaN,5.0,Other,95018,NaN,5663.0,NaN,CRMLSSold202512.csv
2,SanDiego,SanDiego,"Carpet,Wood",True,NaN,NaN,False,2214421.0,1150038683,laura@lauralothianrealestate.com,...,34745.0,NaN,False,3.0,NaN,91364,NaN,34745.0,NaN,CRMLSSold202512.csv
3,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,1200000.0,1150038607,trung.lam@kw.com,...,6600.0,NaN,False,2.0,Other,95121,NaN,6600.0,NaN,CRMLSSold202512.csv
4,TriCounties,TriCounties,NaN,True,NaN,NaN,True,3300.0,1150038314,c21themvpteam@gmail.com,...,17500.0,4.0,False,2.0,Jurupa Unified,92509,0.0,17500.0,NaN,CRMLSSold202512.csv


In [4]:
#Clean column names and rename MLS columns
df.columns = df.columns.str.strip()

df = df.rename(columns={
    "BedroomsTotal": "Bedrooms",
    "BathroomsTotalInteger": "Bathrooms",
    "LotSizeSquareFeet": "LotSize"
})

df.head()



,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSize,MiddleOrJuniorSchoolDistrict,SourceFile
0,ContraCosta,ContraCosta,"Carpet,Tile,Wood",NaN,NaN,NaN,False,1998000.0,1150041639,teresa@teresahooper.com,...,10080.0,NaN,False,3.0,San Ramon Valley,94596,975.0,10080.0,NaN,CRMLSSold202512.csv
1,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,NaN,1150038872,homes@roomsantacruz.com,...,5663.0,NaN,NaN,5.0,Other,95018,NaN,5663.0,NaN,CRMLSSold202512.csv
2,SanDiego,SanDiego,"Carpet,Wood",True,NaN,NaN,False,2214421.0,1150038683,laura@lauralothianrealestate.com,...,34745.0,NaN,False,3.0,NaN,91364,NaN,34745.0,NaN,CRMLSSold202512.csv
3,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,1200000.0,1150038607,trung.lam@kw.com,...,6600.0,NaN,False,2.0,Other,95121,NaN,6600.0,NaN,CRMLSSold202512.csv
4,TriCounties,TriCounties,NaN,True,NaN,NaN,True,3300.0,1150038314,c21themvpteam@gmail.com,...,17500.0,4.0,False,2.0,Jurupa Unified,92509,0.0,17500.0,NaN,CRMLSSold202512.csv


In [5]:
#Create month variable from filename
df["YearMonth"] = df["SourceFile"].str.extract(r"(\d{6})")
df["YearMonth"] = pd.to_datetime(df["YearMonth"], format="%Y%m")

df[["SourceFile", "YearMonth"]].drop_duplicates().sort_values("YearMonth")

,SourceFile,YearMonth
0,CRMLSSold202512.csv,2025-12-01
20538,CRMLSSold202601.csv,2026-01-01
37025,CRMLSSold202602.csv,2026-02-01
55149,CRMLSSold202603.csv,2026-03-01
77732,CRMLSSold202604.csv,2026-04-01
101144,CRMLSSold202605.csv,2026-05-01


In [6]:
#Filter to Residential and SingleFamily Residence
df_filtered = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
].copy()

print("Original rows:", df.shape[0])
print("Filtered rows:", df_filtered.shape[0])

df_filtered.head()

Original rows: 124404
Filtered rows: 61727


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSize,MiddleOrJuniorSchoolDistrict,SourceFile,YearMonth
0,ContraCosta,ContraCosta,"Carpet,Tile,Wood",NaN,NaN,NaN,False,1998000.0,1150041639,teresa@teresahooper.com,...,NaN,False,3.0,San Ramon Valley,94596,975.0,10080.0,NaN,CRMLSSold202512.csv,2025-12-01
2,SanDiego,SanDiego,"Carpet,Wood",True,NaN,NaN,False,2214421.0,1150038683,laura@lauralothianrealestate.com,...,NaN,False,3.0,NaN,91364,NaN,34745.0,NaN,CRMLSSold202512.csv,2025-12-01
3,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,1200000.0,1150038607,trung.lam@kw.com,...,NaN,False,2.0,Other,95121,NaN,6600.0,NaN,CRMLSSold202512.csv,2025-12-01
7,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,3100000.0,1150032869,vickie@realsmartgroup.com,...,NaN,False,1.0,San Jose Unified,95124,NaN,8262.0,NaN,CRMLSSold202512.csv,2025-12-01
9,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,2900000.0,1150028403,vickie@realsmartgroup.com,...,NaN,False,2.0,Other,95128,NaN,9222.0,NaN,CRMLSSold202512.csv,2025-12-01


In [8]:
#Choose target, numerical features, and categorical features
target_col = "ClosePrice"

numeric_features = [
    "LivingArea",
    "Bedrooms",
    "Bathrooms",
    "LotSize",
    "YearBuilt"
]

categorical_features = [
    "CountyOrParish",
    "City",
    "PostalCode"
]

#Check whether these columns exist
all_needed_cols = [target_col] + numeric_features + categorical_features + ["YearMonth"]

for col in all_needed_cols:
    if col not in df_filtered.columns:
        print("Missing column:", col)

In [9]:
#Convert numerical columns to numeric
model_df = df_filtered[all_needed_cols].copy()
print(model_df.shape)
model_df.head()

for col in [target_col] + numeric_features:
    model_df[col] = (
        model_df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
    )
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

model_df.head()

(61727, 10)


,ClosePrice,LivingArea,Bedrooms,Bathrooms,LotSize,YearBuilt,CountyOrParish,City,PostalCode,YearMonth
0,1998000.0,2045.0,4.0,2.0,10080.0,1968.0,Contra Costa,Walnut Creek,94596,2025-12-01
2,2214421.0,3050.0,4.0,4.0,34745.0,1957.0,Los Angeles,Woodland Hills,91364,2025-12-01
3,1200000.0,1594.0,4.0,2.0,6600.0,1978.0,Santa Clara,San Jose,95121,2025-12-01
7,3100000.0,2700.0,5.0,3.0,8262.0,2025.0,Santa Clara,San Jose,95124,2025-12-01
9,2900000.0,2948.0,5.0,4.0,9222.0,2023.0,Santa Clara,San Jose,95128,2025-12-01


In [10]:
#Check missing values, we can get the missing percentage for each variable.
missing_summary = model_df.isna().mean().sort_values(ascending=False) * 100

missing_summary

LotSize           1.751260
YearBuilt         0.058321
LivingArea        0.048601
City              0.022681
Bathrooms         0.001620
PostalCode        0.001620
ClosePrice        0.000000
Bedrooms          0.000000
CountyOrParish    0.000000
YearMonth         0.000000
dtype: float64

In [11]:
#we should drop missing values.
model_df = model_df.dropna(subset=[target_col])

print("Shape after dropping missing ClosePrice:", model_df.shape)

Shape after dropping missing ClosePrice: (61727, 10)


In [12]:
#Create missing-value flags
for col in numeric_features:
    model_df[col + "_missing"] = model_df[col].isna().astype(int)

missing_flag_features = [col + "_missing" for col in numeric_features]

model_df.head()

,ClosePrice,LivingArea,Bedrooms,Bathrooms,LotSize,YearBuilt,CountyOrParish,City,PostalCode,YearMonth,LivingArea_missing,Bedrooms_missing,Bathrooms_missing,LotSize_missing,YearBuilt_missing
0,1998000.0,2045.0,4.0,2.0,10080.0,1968.0,Contra Costa,Walnut Creek,94596,2025-12-01,0,0,0,0,0
2,2214421.0,3050.0,4.0,4.0,34745.0,1957.0,Los Angeles,Woodland Hills,91364,2025-12-01,0,0,0,0,0
3,1200000.0,1594.0,4.0,2.0,6600.0,1978.0,Santa Clara,San Jose,95121,2025-12-01,0,0,0,0,0
7,3100000.0,2700.0,5.0,3.0,8262.0,2025.0,Santa Clara,San Jose,95124,2025-12-01,0,0,0,0,0
9,2900000.0,2948.0,5.0,4.0,9222.0,2023.0,Santa Clara,San Jose,95128,2025-12-01,0,0,0,0,0


In [13]:
#Train/test split by month
available_months = sorted(model_df["YearMonth"].unique())
available_months
test_month = max(available_months)
print("Test month:", test_month)

Test month: 2026-05-01 00:00:00


In [14]:
#Since we have 6 months total, let x=5:
X = 5

train_months = available_months[-(X+1):-1]
print("Training months:", train_months)
print("Test month:", test_month)

train_df = model_df[model_df["YearMonth"].isin(train_months)].copy()
test_df = model_df[model_df["YearMonth"] == test_month].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Training months: [Timestamp('2025-12-01 00:00:00'), Timestamp('2026-01-01 00:00:00'), Timestamp('2026-02-01 00:00:00'), Timestamp('2026-03-01 00:00:00'), Timestamp('2026-04-01 00:00:00')]
Test month: 2026-05-01 00:00:00
Train shape: (49703, 15)
Test shape: (12024, 15)


In [15]:
#Separate X and y
feature_cols = numeric_features + categorical_features + missing_flag_features

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (49703, 13)
X_test: (12024, 13)


In [16]:
#Build preprocessing pipeline
#Numerical missing values: median imputation
#Numerical normalization: StandardScaler
#Categorical missing values: fill with "Missing"
#Categorical encoding: one-hot encoding
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features + missing_flag_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [17]:
#Fit preprocessing on training data only
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed train shape: (49703, 2447)
Processed test shape: (12024, 2447)


In [18]:
#Get processed feature names
numeric_output_features = numeric_features + missing_flag_features

categorical_output_features = (
    preprocessor
    .named_transformers_["cat"]
    .named_steps["encoder"]
    .get_feature_names_out(categorical_features)
)

processed_feature_names = list(numeric_output_features) + list(categorical_output_features)

len(processed_feature_names)

2447

In [19]:
#Convert processed arrays back to DataFrames
X_train_processed_df = pd.DataFrame(
    X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed,
    columns=processed_feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed.toarray() if hasattr(X_test_processed, "toarray") else X_test_processed,
    columns=processed_feature_names,
    index=X_test.index
)

X_train_processed_df.head()

,LivingArea,Bedrooms,Bathrooms,LotSize,YearBuilt,LivingArea_missing,Bedrooms_missing,Bathrooms_missing,LotSize_missing,YearBuilt_missing,...,PostalCode_96093,PostalCode_96094,PostalCode_96097,PostalCode_96101,PostalCode_96106,PostalCode_96114,PostalCode_96130,PostalCode_96137,PostalCode_96143,PostalCode_96150
0,-0.005771,0.521572,-0.564061,-0.021736,-0.300925,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.963625,0.521572,1.198903,-0.020515,-0.697836,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-0.440793,0.521572,-0.564061,-0.021908,0.059903,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.626025,1.553154,0.317421,-0.021826,1.755795,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.865239,1.553154,1.198903,-0.021778,1.683630,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
#Add target and split label
train_clean = X_train_processed_df.copy()
train_clean[target_col] = y_train.values
train_clean["split"] = "train"

test_clean = X_test_processed_df.copy()
test_clean[target_col] = y_test.values
test_clean["split"] = "test"

cleaned_model_data = pd.concat([train_clean, test_clean], axis=0)

print(cleaned_model_data.shape)
cleaned_model_data.head()

(61727, 2449)


,LivingArea,Bedrooms,Bathrooms,LotSize,YearBuilt,LivingArea_missing,Bedrooms_missing,Bathrooms_missing,LotSize_missing,YearBuilt_missing,...,PostalCode_96097,PostalCode_96101,PostalCode_96106,PostalCode_96114,PostalCode_96130,PostalCode_96137,PostalCode_96143,PostalCode_96150,ClosePrice,split
0,-0.005771,0.521572,-0.564061,-0.021736,-0.300925,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1998000.0,train
2,0.963625,0.521572,1.198903,-0.020515,-0.697836,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2214421.0,train
3,-0.440793,0.521572,-0.564061,-0.021908,0.059903,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1200000.0,train
7,0.626025,1.553154,0.317421,-0.021826,1.755795,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3100000.0,train
9,0.865239,1.553154,1.198903,-0.021778,1.683630,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2900000.0,train


In [21]:
#output of final result
cleaned_model_data.to_csv("cleaned_preprocessed_data.csv", index=False)

print("Saved cleaned_preprocessed_data.csv")

Saved cleaned_preprocessed_data.csv


In [22]:
#Just in case, unencoded cleaned dataset
model_df.to_csv("cleaned_unencoded_data.csv", index=False)

print("Saved cleaned_unencoded_data.csv")

Saved cleaned_unencoded_data.csv


In [24]:
pd.read_csv("cleaned_preprocessed_data.csv", nrows=6)

,LivingArea,Bedrooms,Bathrooms,LotSize,YearBuilt,LivingArea_missing,Bedrooms_missing,Bathrooms_missing,LotSize_missing,YearBuilt_missing,...,PostalCode_96097,PostalCode_96101,PostalCode_96106,PostalCode_96114,PostalCode_96130,PostalCode_96137,PostalCode_96143,PostalCode_96150,ClosePrice,split
0,-0.005771,0.521572,-0.564061,-0.021736,-0.300925,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1998000.0,train
1,0.963625,0.521572,1.198903,-0.020515,-0.697836,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2214421.0,train
2,-0.440793,0.521572,-0.564061,-0.021908,0.059903,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1200000.0,train
3,0.626025,1.553154,0.317421,-0.021826,1.755795,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3100000.0,train
4,0.865239,1.553154,1.198903,-0.021778,1.683630,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2900000.0,train
5,-0.528570,-0.510011,-0.564061,-0.022063,0.312483,-0.021517,0.0,-0.004486,-0.134021,-0.024162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1050000.0,train
